In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
!{sys.executable} -m pip uninstall torchvision datasets transformers torch scikit-learn -y
!{sys.executable} -m pip install torch transformers datasets scikit-learn -q
!{sys.executable} -m pip install torchvision -q

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 5.10.1
Uninstalling transformers-5.10.1:
  Successfully uninstalled transformers-5.10.1
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


*   BASE_PATH → Drive'daki veri klasörünün yolu
*   MODEL_NAME → Kullanacağımız BERT modeli (DistilBERT)
*   RANDOM_STATE → Rastgeleliği sabitlemek için (herkes aynı sonucu alır)

*   MAX_LENGTH → Her review'ın tokenize edilirken kesilebileceği maksimum kelime sayısı
*   LABEL2ID / ID2LABEL → Sınıf isimlerini sayıya, sayıları tekrar isme çeviren sözlükler (model sayılarla çalışır)





In [5]:
BASE_PATH = "/content/drive/MyDrive/amazon-customer-review/data/"
MODEL_NAME = "distilbert-base-uncased"
RANDOM_STATE = 42
MAX_LENGTH = 128

LABEL2ID = {
    'problem_yok': 0,
    'ürün_kalitesi': 1,
    'ürün_dayanıklılığı': 2,
    'performans': 3,
    'içerik_beklenti': 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [6]:
df = pd.read_csv(BASE_PATH + "labeled_data.csv")
print(f"Shape: {df.shape}")
print(df['problem_category'].value_counts())

Shape: (116681, 7)
problem_category
problem_yok           106619
ürün_kalitesi           4216
ürün_dayanıklılığı      2525
performans              1857
içerik_beklenti         1464
Name: count, dtype: int64


In [7]:
print(df.columns.tolist())
print(df.head(2))

['review_headline', 'review_body', 'star_rating', 'verified_purchase', 'helpful_votes', 'total_votes', 'problem_category']
  review_headline                                        review_body  \
0        One Star  Can't finde satelite for DECTVHD Had to get a ...   
1        One Star  I just got them last week. They don't get full...   

   star_rating verified_purchase  helpful_votes  total_votes  \
0            1                 Y              0            0   
1            1                 Y              1            1   

     problem_category  
0  ürün_dayanıklılığı  
1       ürün_kalitesi  


ciddi bir class imbalance var:

problem_yok domine eden çoğunlukta — BERT bunu düzeltmeden sadece problem_yok tahmin eder ve %91 accuracy gösterir ama domine eden class bu olduğu için anlamlı olmaz.

Bu sebeple Undersample + class weight yapacağız

In [8]:
# Class Imbalance — Her sınıftan max 2000 örnek
dfs = []
for label in df['problem_category'].unique():
    subset = df[df['problem_category'] == label]
    if len(subset) >= 2000:
        subset = subset.sample(n=2000, random_state=RANDOM_STATE)
    dfs.append(subset)

df_balanced = pd.concat(dfs).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(df_balanced['problem_category'].value_counts())

problem_category
ürün_dayanıklılığı    2000
ürün_kalitesi         2000
problem_yok           2000
performans            1857
içerik_beklenti       1464
Name: count, dtype: int64


In [9]:
# Label Encoding
df_balanced['label'] = df_balanced['problem_category'].map(LABEL2ID)

# Train / Val / Test Split
train_df, test_df = train_test_split(df_balanced, test_size=0.2, stratify=df_balanced['label'], random_state=RANDOM_STATE)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=RANDOM_STATE)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 6710, Val: 746, Test: 1865


In [10]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# HuggingFace Dataset formatına çevir
def tokenize(batch):
    return tokenizer(batch['review_body'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

train_dataset = Dataset.from_pandas(train_df[['review_body', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['review_body', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['review_body', 'label']].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization tamamlandı.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6710 [00:00<?, ? examples/s]

Map:   0%|          | 0/746 [00:00<?, ? examples/s]

Map:   0%|          | 0/1865 [00:00<?, ? examples/s]

Tokenization tamamlandı.


In [11]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Training Arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/amazon-customer-review/model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    logging_steps=50,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True,
)

print("Model ve training arguments hazır.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model ve training arguments hazır.


In [12]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = (preds == labels).mean()
    return {'accuracy': acc, 'f1': f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer hazır.")

Trainer hazır.


In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.482638,0.505811,0.828418,0.828171
2,0.321755,0.494565,0.843164,0.842486
3,0.119997,0.541370,0.855228,0.855372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1260, training_loss=0.40959968912223027, metrics={'train_runtime': 218.0115, 'train_samples_per_second': 92.335, 'train_steps_per_second': 5.78, 'total_flos': 666677849587200.0, 'train_loss': 0.40959968912223027, 'epoch': 3.0})

In [14]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.119997,0.529814,3,0.838070,0.836562


{'eval_loss': 0.5298137664794922, 'eval_accuracy': 0.8380697050938338, 'eval_f1': 0.8365615366540685}


In [15]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(predictions.label_ids, preds, target_names=list(LABEL2ID.keys())))

                    precision    recall  f1-score   support

       problem_yok       0.89      0.85      0.87       400
     ürün_kalitesi       0.80      0.82      0.81       400
ürün_dayanıklılığı       0.85      0.71      0.77       400
        performans       0.80      0.93      0.86       372
   içerik_beklenti       0.86      0.90      0.88       293

          accuracy                           0.84      1865
         macro avg       0.84      0.84      0.84      1865
      weighted avg       0.84      0.84      0.84      1865



In [16]:
model.save_pretrained("/content/drive/MyDrive/amazon-customer-review/model/bert_final")
tokenizer.save_pretrained("/content/drive/MyDrive/amazon-customer-review/model/bert_final")
print("Model kaydedildi.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model kaydedildi.
